In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies = pd.read_csv("/content/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
movies.shape

(9742, 3)

In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [ ]:
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [ ]:
movies.isnull().sum()

,0
movieId,0
title,0
genres,0


In [ ]:
movies.duplicated().sum()

np.int64(0)

In [ ]:
movies["genres"].head(20)

,genres
0,Adventure|Animation|Children|Comedy|Fantasy
1,Adventure|Children|Fantasy
2,Comedy|Romance
3,Comedy|Drama|Romance
4,Comedy
5,Action|Crime|Thriller
6,Comedy|Romance
7,Adventure|Children
8,Action
9,Action|Adventure|Thriller


In [ ]:
movies = movies.drop_duplicates()

movies = movies.dropna(subset=["title", "genres"])

movies = movies.reset_index(drop=True)

In [ ]:
movies.isnull().sum()

,0
movieId,0
title,0
genres,0


In [ ]:
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

In [ ]:
movies[["title", "genres"]].head()

,title,genres
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy Romance
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy


In [ ]:
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies["genres"])

In [ ]:
tfidf_matrix.shape

(9742, 23)

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [ ]:
cosine_sim.shape

(9742, 9742)

In [ ]:
indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

In [ ]:
indices["Toy Story (1995)"]

np.int64(0)

In [ ]:
def recommend_movies(title, num_recommendations=10):

    if title not in indices:
        return pd.DataFrame(columns=["title", "genres"])

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:num_recommendations + 1]

    movie_indices = [i[0] for i in similarity_scores]

    recommendations = movies.iloc[movie_indices][
        ["title", "genres"]
    ].copy()

    recommendations["similarity_score"] = [
        round(i[1], 4)
        for i in similarity_scores
    ]

    return recommendations

In [ ]:
recommend_movies("Toy Story (1995)")

,title,genres,similarity_score
1706,Antz (1998),Adventure Animation Children Comedy Fantasy,1.0
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3000,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.0
3568,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.0
6194,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy,1.0
6486,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy,1.0
6948,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy,1.0
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure Animation Children Comedy Fantasy,1.0
8219,Turbo (2013),Adventure Animation Children Comedy Fantasy,1.0


In [ ]:
recommend_movies("Jumanji (1995)")

,title,genres,similarity_score
53,"Indian in the Cupboard, The (1995)",Adventure Children Fantasy,1.0
109,"NeverEnding Story III, The (1994)",Adventure Children Fantasy,1.0
767,Escape to Witch Mountain (1975),Adventure Children Fantasy,1.0
1514,Darby O'Gill and the Little People (1959),Adventure Children Fantasy,1.0
1556,Return to Oz (1985),Adventure Children Fantasy,1.0
1617,"NeverEnding Story, The (1984)",Adventure Children Fantasy,1.0
1618,"NeverEnding Story II: The Next Chapter, The (1...",Adventure Children Fantasy,1.0
1799,Santa Claus: The Movie (1985),Adventure Children Fantasy,1.0
3574,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure Children Fantasy,1.0
6075,"Chronicles of Narnia: The Lion, the Witch and ...",Adventure Children Fantasy,1.0


In [ ]:
recommend_movies("The Matrix (1999)")

,title,genres


In [ ]:
!pip install joblib

In [ ]:
import joblib

In [ ]:
joblib.dump(
    movies,
    "movies.pkl"
)

joblib.dump(
    tfidf,
    "tfidf.pkl"
)

joblib.dump(
    cosine_sim,
    "cosine_similarity.pkl"
)

joblib.dump(
    indices,
    "movie_indices.pkl"
)

['movie_indices.pkl']

In [ ]:
import os

os.listdir()

['.config',
 'movies.csv',
 'cosine_similarity.pkl',
 'tfidf.pkl',
 'movie_indices.pkl',
 'movies.pkl',
 'sample_data']

In [ ]:
import joblib


movies = joblib.load("movies.pkl")
cosine_sim = joblib.load("cosine_similarity.pkl")
indices = joblib.load("movie_indices.pkl")


def recommend_movies(title, num_recommendations=10):

    if title not in indices:
        return []

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:num_recommendations + 1
    ]

    recommendations = []

    for movie_index, score in similarity_scores:

        recommendations.append({
            "title": movies.iloc[movie_index]["title"],
            "genres": movies.iloc[movie_index]["genres"],
            "similarity": round(float(score), 4)
        })

    return recommendations

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.1 MB/s eta 0:00:00


In [ ]:
import streamlit as st
import joblib

print("Streamlit installed successfully!")

Streamlit installed successfully!


In [ ]:
%%writefile app.py

st.set_page_config(
    page_title="Movie Recommendation System",
    page_icon="🎬",
    layout="wide"
)


@st.cache_resource
def load_model():

    movies = joblib.load("movies.pkl")
    cosine_sim = joblib.load("cosine_similarity.pkl")
    indices = joblib.load("movie_indices.pkl")

    return movies, cosine_sim, indices


movies, cosine_sim, indices = load_model()


def recommend_movies(title, num_recommendations):

    if title not in indices:
        return []

    idx = indices[title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:num_recommendations + 1
    ]

    recommendations = []

    for movie_index, score in similarity_scores:

        recommendations.append({
            "title": movies.iloc[movie_index]["title"],
            "genres": movies.iloc[movie_index]["genres"],
            "similarity": float(score)
        })

    return recommendations


st.title("Movie Recommendation System")

st.write(
    """
    This recommendation system uses **Content-Based Filtering**
    to recommend movies based on genre similarity.
    """
)

st.divider()


st.sidebar.header("Recommendation Settings")

num_recommendations = st.sidebar.slider(
    "Number of recommendations",
    min_value=5,
    max_value=20,
    value=10
)


movie_list = sorted(
    movies["title"].tolist()
)

selected_movie = st.selectbox(
    "Select a movie",
    movie_list
)



if st.button("Get Recommendations"):

    recommendations = recommend_movies(
        selected_movie,
        num_recommendations
    )

    st.subheader(
        f"Recommendations based on: {selected_movie}"
    )

    if recommendations:

        for i, movie in enumerate(
            recommendations,
            start=1
        ):

            st.markdown(
                f"""
                ### {i}. {movie['title']}

                **Genres:** {movie['genres']}

                **Similarity Score:** {movie['similarity']:.2f}

                ---
                """
            )

    else:

        st.warning(
            "Sorry, no recommendations were found."
        )


st.sidebar.divider()

st.sidebar.info(
    """
    **Project:** Movie Recommendation System

    **Method:** Content-Based Filtering

    **Algorithm:** TF-IDF + Cosine Similarity

    **Dataset:** MovieLens
    """
)

Writing app.py


In [35]:
def precision_recall_at_k(
    movie_index,
    k=10
):

    similarity_scores = list(
        enumerate(cosine_sim[movie_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommended_indices = [
        x[0]
        for x in similarity_scores[1:k + 1]
    ]

    original_genres = set(
        movies.iloc[movie_index]["genres"].split()
    )

    relevant_movies = []

    for idx in recommended_indices:

        recommended_genres = set(
            movies.iloc[idx]["genres"].split()
        )

        if original_genres.intersection(
            recommended_genres
        ):
            relevant_movies.append(idx)

    relevant_count = len(relevant_movies)

    precision = relevant_count / k

    total_relevant = 0

    for idx in range(len(movies)):

        if idx == movie_index:
            continue

        movie_genres = set(
            movies.iloc[idx]["genres"].split()
        )

        if original_genres.intersection(
            movie_genres
        ):
            total_relevant += 1

    if total_relevant > 0:
        recall = relevant_count / total_relevant
    else:
        recall = 0

    return precision, recall

In [36]:
precisions = []
recalls = []

for i in range(min(100, len(movies))):

    precision, recall = precision_recall_at_k(
        i,
        k=10
    )

    precisions.append(precision)
    recalls.append(recall)

In [37]:
average_precision = np.mean(precisions)
average_recall = np.mean(recalls)

print(
    "Average Precision@10:",
    round(average_precision, 4)
)

print(
    "Average Recall@10:",
    round(average_recall, 4)
)

Average Precision@10: 1.0
Average Recall@10: 0.003


In [38]:
evaluation_results = pd.DataFrame({
    "Metric": [
        "Precision@10",
        "Recall@10"
    ],
    "Score": [
        average_precision,
        average_recall
    ]
})

evaluation_results

,Metric,Score
0,Precision@10,1.000000
1,Recall@10,0.002972
